# DataDiff Pro — Batch Comparison Notebook

Use this notebook to compare large files (JSON, XML, CSV) without the browser hanging.

## How to use

1. **Edit Cell 2 (Config)** — fill in your file paths, format, and batch settings.
2. Click **Kernel → Restart & Run All** (or run cells one by one from top to bottom).
3. Each run compares **one batch** of records starting at `batch_start`.
4. To compare the next batch, change `batch_start` in Cell 2 and run again.
5. Results are saved to the CSV and JSON files you define in `output_csv_file` / `output_json_file`.

## Batch workflow example (10 000 records, batch_size = 500)

| Run | batch_start | Records compared |
|-----|-------------|------------------|
| 1   | 0           | 0 – 499          |
| 2   | 500         | 500 – 999        |
| 3   | 1000        | 1000 – 1499      |
| … | … | … |
| 20  | 9500        | 9500 – 9999      |

## Source / Target options

- **Local file** — set `source_file` to the absolute path of your file.
- **API URL** — set `source_api_url` to a REST endpoint that returns JSON/XML/CSV and set `source_file = None`.

> **No changes to the existing app** — this notebook imports the Python core directly and is fully independent.

In [ ]:
# =============================================================
# CELL 2 — CONFIG
# This is the ONLY cell you need to edit.
# Run all cells after saving your changes.
# =============================================================

# --- Source (left side) ---
source_file        = "temp/source.json"   # set to None to use API URL instead
source_api_url     = None                 # e.g. "https://api.example.com/data"
source_api_headers = {}                   # e.g. {"Authorization": "Bearer mytoken"}

# --- Target (right side) ---
target_file        = "temp/target.json"
target_api_url     = None
target_api_headers = {}

# --- Format ---
left_format  = "json"   # "json", "xml", or "csv"
right_format = "json"

# --- Field mapping (leave as empty string if not needed) ---
# One rule per line: source_field,target_field
# Lines starting with # are ignored.
mapper_csv = """
user_id,uid
profile.name,profile_info.full_name
profile.email,profile_info.email_address
orders.order_id,orders.id
orders.amount,orders.value
"""

# --- Environment YAML: equivalence rules and list keys ---
# Leave as empty string if not needed.
environment_yaml = """
list_keys:
  records: [uid]
  orders: [id]
"""

# --- Advanced options ---
mapper_direction = "left_to_right"   # "left_to_right" or "right_to_left"
strict_mode      = False             # True = error if a mapped field is missing
multi_match_rule = "keep_list"       # "keep_list", "flatten", or "pick_first"
deep_mode        = False             # True = expand string-encoded JSON/XML values

# When True: a field that is null/empty on one side and missing on the other
# is treated as EQUIVALENT instead of EXTRA_LEFT / EXTRA_RIGHT.
null_missing_equivalent = False

# --- Batch settings ---
batch_start = 0     # Record index to start from (0 = beginning, 500 = skip first 500)
batch_size  = 500   # How many records to compare in this run
records_key = None  # Set to e.g. "data" if records are wrapped: {"data": [...]}
                    # Leave as None to auto-detect.

# --- ID / Key fields for KEY_MISSING pre-check ---
# If a record is missing its ID field entirely, it can't be identified —
# skip it immediately instead of running a full diff.
# Set both to None to disable the pre-check (default, backwards compatible).
source_id_field = "user_id"   # e.g. "id", "recordId", "sku"
target_id_field = "uid"       # e.g. "id", "record_id", "sku"

# --- Output files ---
output_csv_file  = "results/comparison_results.csv"
output_json_file = "results/comparison_results.json"

# --- Export filter ---
# Control which status types are written to the output files.
# Set to None to export everything (default).
# Available statuses: "MATCH", "MISMATCH", "EQUIVALENT", "EXTRA_LEFT", "EXTRA_RIGHT", "KEY_MISSING"
export_statuses = None   # e.g. ["MISMATCH", "EXTRA_LEFT", "EXTRA_RIGHT", "KEY_MISSING"]

# Control which columns appear in the CSV.
# Set to None to export all columns (default).
# Available columns: "batch_start", "path", "left_value", "right_value", "status",
#                    "equivalence_rule", "list_strategy", "type_mismatch", "list_key_values"
export_columns = None    # e.g. ["path", "left_value", "right_value", "status"]

# =============================================================
print("Config loaded.")

In [ ]:
# CELL 3 — SETUP: imports and sys.path
# No changes needed here.

import sys
import os
import json
import csv
import datetime

# Allow importing from core/ (one level up from notebook/)
_PROJECT_ROOT = os.path.abspath(os.path.join(os.path.dirname("__file__"), ".."))
if _PROJECT_ROOT not in sys.path:
    sys.path.insert(0, _PROJECT_ROOT)

from core.normalizer import normalize
from core.mapper import parse_mapping, apply_mapping
from core.equivalence import EquivalenceEngine
from core.list_resolver import ListResolver
from core.diff_engine import DiffEngine
from core.deep_expander import deep_expand

try:
    import requests
    _REQUESTS_AVAILABLE = True
except ImportError:
    _REQUESTS_AVAILABLE = False
    print("WARNING: 'requests' not installed. API URL fetching will not work.")
    print("         Fix: pip install requests")

print("Imports OK.")

In [4]:
# CELL 4 — LOAD DATA from file or API URL
# No changes needed here.

def _load_file(path):
    """Read a local file and return its text content."""
    with open(path, "r", encoding="utf-8") as f:
        return f.read()

def _load_api(url, headers, timeout=30):
    """Fetch data from an API URL and return the response body as text."""
    if not _REQUESTS_AVAILABLE:
        raise RuntimeError(
            "'requests' library is required for API fetching. "
            "Run: pip install requests"
        )
    resp = requests.get(url, headers=headers or {}, timeout=timeout)
    resp.raise_for_status()
    return resp.text

# Load source
if source_file:
    print(f"Loading source from file: {source_file}")
    source_raw = _load_file(source_file)
elif source_api_url:
    print(f"Fetching source from API: {source_api_url}")
    source_raw = _load_api(source_api_url, source_api_headers)
else:
    raise ValueError(
        "No source defined. Set source_file or source_api_url in the Config cell."
    )

# Load target
if target_file:
    print(f"Loading target from file: {target_file}")
    target_raw = _load_file(target_file)
elif target_api_url:
    print(f"Fetching target from API: {target_api_url}")
    target_raw = _load_api(target_api_url, target_api_headers)
else:
    raise ValueError(
        "No target defined. Set target_file or target_api_url in the Config cell."
    )

print(f"Source: {len(source_raw):,} characters")
print(f"Target: {len(target_raw):,} characters")

Loading source from file: temp/source.json
Loading target from file: temp/target.json
Source: 1,222 characters
Target: 1,142 characters


In [ ]:
# CELL 5 — PARSE + VALIDATE
# No changes needed here.

# Parse both sides using the same normalizer as the web API
left_data, left_err = normalize(source_raw, left_format)
if left_err:
    raise ValueError(f"Source parse error: {left_err}")

right_data, right_err = normalize(target_raw, right_format)
if right_err:
    raise ValueError(f"Target parse error: {right_err}")

# Extract the list of records from the parsed data
def _extract_record_list(data, key=None):
    """Return the list of records from parsed data."""
    if isinstance(data, list):
        return data
    if isinstance(data, dict):
        if key and key in data:
            candidate = data[key]
            if isinstance(candidate, list):
                return candidate
            raise ValueError(f"records_key='{key}' found but its value is not a list.")
        for k, v in data.items():
            if isinstance(v, list) and v and isinstance(v[0], dict):
                print(f"  Auto-detected record list under key: '{k}'")
                return v
    raise ValueError(
        "Cannot find a list of records in the data. "
        "Set records_key in the Config cell to the key that wraps your records "
        "(e.g. records_key = 'data')."
    )

left_records  = _extract_record_list(left_data,  records_key)
right_records = _extract_record_list(right_data, records_key)

print(f"Source total records: {len(left_records):,}")
print(f"Target total records: {len(right_records):,}")

# Parse field mapping (done once, applied per batch)
_mapping_pairs = []
if mapper_csv and mapper_csv.strip():
    _mapping_pairs, map_err = parse_mapping(mapper_csv)
    if map_err:
        raise ValueError(f"Field mapping error: {map_err}")
    print(f"Field mapping: {len(_mapping_pairs)} rule(s) from mapper_csv")

# --- Auto-inject record key config from source_id_field / target_id_field ---
# Mirrors the UI's Source/Target Record Key inputs:
#   1. If keys differ, add the rename rule to _mapping_pairs (if not already there)
#   2. Inject list_keys: records: [target_key] into environment_yaml (if not already set)
_yaml = environment_yaml or ""

if source_id_field or target_id_field:
    _src_k = (source_id_field or "").strip()
    _tgt_k = (target_id_field or "").strip()
    _resolved_k = _tgt_k or _src_k  # post-mapping key name is the target key

    # 1. Auto-add rename rule to mapping if keys differ
    if _src_k and _tgt_k and _src_k != _tgt_k:
        _rule = (_src_k, _tgt_k)
        if _rule not in _mapping_pairs:
            _mapping_pairs.insert(0, _rule)  # prepend so it runs first
            print(f"Auto-injected key mapping rule: {_src_k} → {_tgt_k}")

    # 2. Auto-inject list_keys into YAML if not already present
    if _resolved_k and "list_keys:" not in _yaml:
        _key_yaml = f"\nlist_keys:\n  records: [{_resolved_k}]"
        _yaml = _yaml + _key_yaml
        print(f"Auto-injected list_keys: records: [{_resolved_k}]")
# --- end auto-inject ---

if not _mapping_pairs:
    print("Field mapping: none")

# Build engines once — they are stateless after __init__
_eq_engine     = EquivalenceEngine(_yaml)
_list_resolver = ListResolver(_yaml)
_engine        = DiffEngine(_eq_engine, _list_resolver, deep_mode=deep_mode,
                             null_missing_equivalent=null_missing_equivalent)

print("Pipeline ready.")

In [ ]:
# CELL 6 — COMPARE ONE BATCH
# No changes needed here.
# Change batch_start in the Config cell and re-run from Cell 6 for the next batch.

_batch_end = batch_start + batch_size

left_batch  = left_records[batch_start:_batch_end]
right_batch = right_records[batch_start:_batch_end]

if not left_batch and not right_batch:
    raise ValueError(
        f"batch_start={batch_start} is beyond the end of both files. "
        f"Source has {len(left_records):,} records, "
        f"target has {len(right_records):,} records."
    )

_actual_end = min(_batch_end, max(len(left_records), len(right_records)))
print(
    f"Comparing records {batch_start} to {_actual_end - 1}  "
    f"({len(left_batch)} source / {len(right_batch)} target)"
)

# Deep expand BEFORE mapping so the mapper can navigate into
# string-encoded JSON/XML fields (e.g. "profile": "{\"name\":\"Alice\"}")
_left_to_diff  = deep_expand(left_batch)  if deep_mode else list(left_batch)
_right_to_diff = deep_expand(right_batch) if deep_mode else list(right_batch)

# Apply field mapping to this batch
if _mapping_pairs:
    _direction = (mapper_direction or "left_to_right").strip().lower()
    if _direction == "left_to_right":
        _left_to_diff, _ = apply_mapping(
            _left_to_diff, _mapping_pairs,
            strict_mode=strict_mode,
            multi_match_rule=multi_match_rule,
        )
    elif _direction == "right_to_left":
        _right_to_diff, _ = apply_mapping(
            _right_to_diff, _mapping_pairs,
            strict_mode=strict_mode,
            multi_match_rule=multi_match_rule,
        )
    else:
        raise ValueError(
            f"Invalid mapper_direction '{mapper_direction}'. "
            "Use 'left_to_right' or 'right_to_left'."
        )

# --- KEY_MISSING pre-check ---
# Runs AFTER mapping, so both sides are on the target schema.
# Use target_id_field to check both sides (left was renamed, right was always target).
_key_missing_records = []
_post_map_key = (target_id_field or source_id_field)  # post-mapping field name

if _post_map_key is not None:
    _filtered_left  = []
    _filtered_right = []
    _batch_len = max(len(_left_to_diff), len(_right_to_diff))

    for _i in range(_batch_len):
        _l_rec = _left_to_diff[_i]  if _i < len(_left_to_diff)  else None
        _r_rec = _right_to_diff[_i] if _i < len(_right_to_diff) else None

        # After mapping both sides use the target field name
        _l_missing = isinstance(_l_rec, dict) and _post_map_key not in _l_rec
        _r_missing = isinstance(_r_rec, dict) and _post_map_key not in _r_rec

        if _l_missing or _r_missing:
            # Best-effort path label
            _id_val = None
            if not _l_missing and isinstance(_l_rec, dict):
                _id_val = _l_rec.get(_post_map_key)
            elif not _r_missing and isinstance(_r_rec, dict):
                _id_val = _r_rec.get(_post_map_key)
            _record_id_label = (
                f"{_post_map_key}={_id_val}"
                if _id_val is not None
                else f"index={batch_start + _i}"
            )

            _reason_parts = []
            if _l_missing:
                _reason_parts.append(f"source missing '{_post_map_key}'")
            if _r_missing:
                _reason_parts.append(f"target missing '{_post_map_key}'")

            _key_missing_records.append({
                "batch_start":      batch_start,
                "record_id":        _record_id_label,
                "path":             "(record)",
                "left_value":       str(_l_rec) if _l_rec is not None else None,
                "right_value":      str(_r_rec) if _r_rec is not None else None,
                "status":           "KEY_MISSING",
                "equivalence_rule": "",
                "list_strategy":    "",
                "type_mismatch":    "",
                "list_key_values":  ", ".join(_reason_parts),
            })
        else:
            if _l_rec is not None:
                _filtered_left.append(_l_rec)
            if _r_rec is not None:
                _filtered_right.append(_r_rec)

    _left_to_diff  = _filtered_left
    _right_to_diff = _filtered_right

    if _key_missing_records:
        print(f"KEY_MISSING pre-check: {len(_key_missing_records)} record(s) skipped.")
# --- end KEY_MISSING pre-check ---

# Wrap in dict so list_keys: records: [uid] in the YAML matches the "records" key.
# Without this the engine compares a bare list and uses "(root)" as the list name,
# which never matches any list_keys config → auto-detect picks a wrong field.
_diff_result = _engine.compare({"records": _left_to_diff}, {"records": _right_to_diff})

# Flatten DiffRecord objects into plain dicts for easy export.
# Strip the "records" prefix added by the dict-wrapping above so paths look clean,
# and extract the record_id into a dedicated column.
def _split_record_path(full_path: str):
    """
    Split 'records[uid=101].profile_info.name' into:
      record_id = 'uid=101'
      field_path = 'profile_info.name'
    Also strips the leading 'records' wrapper key.
    """
    path = full_path
    # Strip leading "records" key (added by wrapping)
    if path.startswith("records"):
        path = path[len("records"):]  # now starts with "[uid=101]...." or "(root)"

    record_id = ""
    field_path = path

    if path.startswith("["):
        end = path.find("]")
        if end != -1:
            record_id = path[1:end]         # "uid=101"
            rest = path[end + 1:]
            field_path = rest.lstrip(".")   # "profile_info.name"
    return record_id, field_path

_all_records = []
for _rec in _diff_result.records:
    _record_id, _field_path = _split_record_path(_rec.path)
    _row = {
        "batch_start":      batch_start,
        "record_id":        _record_id,
        "path":             _field_path,
        "left_value":       _rec.left_value,
        "right_value":      _rec.right_value,
        "status":           _rec.status,
        "equivalence_rule": "",
        "list_strategy":    "",
        "type_mismatch":    "",
        "list_key_values":  "",
    }
    if _rec.trace:
        _row["equivalence_rule"] = _rec.trace.get("equivalence_rule", "")
        _row["list_strategy"]    = _rec.trace.get("list_strategy", "")
        _row["type_mismatch"]    = _rec.trace.get("type_mismatch", "")
        _row["list_key_values"]  = str(_rec.trace.get("list_key_values", ""))
    _all_records.append(_row)

# Prepend KEY_MISSING rows so they appear first
_all_records = _key_missing_records + _all_records

print(
    f"Done. {len(_all_records) - len(_key_missing_records):,} field comparisons"
    + (f" + {len(_key_missing_records)} KEY_MISSING record(s) skipped." if _key_missing_records else " completed.")
)

In [ ]:
# CELL 7 — SUMMARY TABLE
# No changes needed here.

_summary = _diff_result.summary   # dict: MATCH, MISMATCH, EQUIVALENT, EXTRA_LEFT, EXTRA_RIGHT
_total   = sum(_summary.values())

print("=" * 55)
print(f"  BATCH SUMMARY  (records {batch_start} – {_actual_end - 1})")
print("=" * 55)
print(f"  {'Status':<20} {'Count':>10} {'Percent':>10}")
print("-" * 55)
for _status, _count in _summary.items():
    _pct = (_count / _total * 100) if _total > 0 else 0.0
    print(f"  {_status:<20} {_count:>10,} {_pct:>9.1f}%")
print("-" * 55)
print(f"  {'TOTAL FIELDS':<20} {_total:>10,}")
print("=" * 55)

_issues = [r for r in _all_records if r["status"] in ("MISMATCH", "EXTRA_LEFT", "EXTRA_RIGHT", "KEY_MISSING")]
print(f"\nIssues (MISMATCH / EXTRA / KEY_MISSING) in this batch: {len(_issues):,}")

if _issues:
    _preview = _issues[:20]
    print(f"\nFirst {len(_preview)} issues:")
    print(f"  {'Record ID':<25} {'Field Path':<35} {'Status':<12} {'Source':>15} {'Target':>15}")
    print(f"  {'-'*25} {'-'*35} {'-'*12} {'-'*15} {'-'*15}")
    for _r in _preview:
        _rid  = str(_r.get("record_id", ""))[:25]
        _p    = str(_r["path"])[:35]
        _s    = str(_r["status"])[:12]
        _l    = str(_r["left_value"])[:15]
        _rv   = str(_r["right_value"])[:15]
        print(f"  {_rid:<25} {_p:<35} {_s:<12} {_l:>15} {_rv:>15}")

In [ ]:
# CELL 8 — EXPORT RESULTS
# No changes needed here.
# Results are appended to the output files so you can collect all batches in one file.

import pathlib

_csv_path  = pathlib.Path(output_csv_file)
_json_path = pathlib.Path(output_json_file)
_csv_path.parent.mkdir(parents=True, exist_ok=True)
_json_path.parent.mkdir(parents=True, exist_ok=True)

_CSV_COLUMNS = [
    "batch_start", "record_id", "path", "left_value", "right_value", "status",
    "equivalence_rule", "list_strategy", "type_mismatch", "list_key_values",
]

# Apply export filters (status filter + column filter from Config cell)
_rows_to_export = _all_records
if export_statuses is not None:
    _rows_to_export = [r for r in _rows_to_export if r["status"] in export_statuses]

_cols_to_export = export_columns if export_columns is not None else _CSV_COLUMNS
# Always keep only known columns (guard against typos in config)
_cols_to_export = [c for c in _cols_to_export if c in _CSV_COLUMNS]

# Append to CSV (write header only if file does not exist yet)
_csv_is_new = not _csv_path.exists()
with open(_csv_path, "a", newline="", encoding="utf-8") as _f:
    _writer = csv.DictWriter(_f, fieldnames=_cols_to_export, extrasaction="ignore")
    if _csv_is_new:
        _writer.writeheader()
    _writer.writerows(_rows_to_export)

_csv_mode = "created" if _csv_is_new else "appended to"
print(f"CSV {_csv_mode}: {_csv_path}  ({len(_rows_to_export):,} rows added)")

# Write / overwrite JSON for this batch
_run_ts = datetime.datetime.now().isoformat()
_json_payload = {
    "run_metadata": {
        "source":              source_file or source_api_url,
        "target":              target_file or target_api_url,
        "left_format":         left_format,
        "right_format":        right_format,
        "batch_start":         batch_start,
        "batch_size":          batch_size,
        "records_in_batch":    len(left_batch),
        "total_source_records": len(left_records),
        "total_target_records": len(right_records),
        "run_timestamp":       _run_ts,
    },
    "summary": _summary,
    "records": _rows_to_export,
}
with open(_json_path, "w", encoding="utf-8") as _f:
    json.dump(_json_payload, _f, indent=2, default=str)

print(f"JSON saved: {_json_path}")
print()
print("Done!")
print(f"To compare the next batch, set  batch_start = {batch_start + batch_size}  in the Config cell and run again.")